In [2]:
# ============================================================
# 03_frame_extraction.ipynb
# TRIMMED FRAME EXTRACTION FOR RGB + THERMAL(NIR)
# + OPTIONAL RGB BACKGROUND REMOVAL
# ============================================================
# Creates:
# 1. data/processed/frames_trimmed/RGB/<distance>/<gesture>/<subject_id>/<pair_id>/
# 2. data/processed/frames_trimmed/NIR/<distance>/<gesture>/<subject_id>/<pair_id>/
# 3. data/processed/frames_trimmed/RGB_BGREM/<distance>/<gesture>/<subject_id>/<pair_id>/
#
# Trimming rule:
# remove 1 second from start and 1 second from end
#
# Notes:
# - RGB_bg_removed is optional but enabled here
# - Keep original trimmed RGB too
# - Do NOT use only bg-removed frames as final truth without comparison
# ============================================================

import os
import json
import time
import shutil
import warnings
import subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)

# ---------------------------
# Optional background removal
# ---------------------------
USE_RGB_BACKGROUND_REMOVAL = True

try:
    from rembg import remove, new_session
    REMBG_AVAILABLE = True
except Exception:
    REMBG_AVAILABLE = False
    USE_RGB_BACKGROUND_REMOVAL = False

# ============================================================
# 1. PATHS
# ============================================================
cwd = Path.cwd()
if cwd.name == "notebooks":
    ROOT = cwd.parent
else:
    ROOT = cwd

MANIFEST_PATH = ROOT / "data" / "processed" / "manifests" / "paired_manifest_with_subjects.csv"
FRAMES_ROOT = ROOT / "data" / "processed" / "frames_trimmed"
MANIFEST_DIR = ROOT / "data" / "processed" / "manifests"

FRAMES_ROOT.mkdir(parents=True, exist_ok=True)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

SUMMARY_CSV = MANIFEST_DIR / "frame_extraction_trimmed_summary.csv"
SUMMARY_JSON = MANIFEST_DIR / "frame_extraction_trimmed_summary.json"

print("ROOT:", ROOT)
print("MANIFEST_PATH:", MANIFEST_PATH)
print("FRAMES_ROOT:", FRAMES_ROOT)
print("USE_RGB_BACKGROUND_REMOVAL:", USE_RGB_BACKGROUND_REMOVAL)

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Manifest not found: {MANIFEST_PATH}")

# ============================================================
# 2. SETTINGS
# ============================================================
LIMIT_ROWS = None          # set 10 for quick test
FORCE_REEXTRACT = False
SKIP_IF_ALREADY_DONE = True

TRIM_START_SEC = 1.0
TRIM_END_SEC = 1.0

JPEG_QUALITY = 95
MAX_WORKERS = 8

# ============================================================
# 3. LOAD MANIFEST
# ============================================================
df = pd.read_csv(MANIFEST_PATH)

required_cols = [
    "pair_id", "subject_id", "gesture", "distance",
    "rgb_video_path", "nir_video_path", "is_valid_pair"
]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

df = df[df["is_valid_pair"] == True].copy().reset_index(drop=True)

if LIMIT_ROWS is not None:
    df = df.head(LIMIT_ROWS).copy()

print("Rows to process:", len(df))
display(df.head())

# ============================================================
# 4. HELPERS
# ============================================================
def safe_str(x):
    if pd.isna(x):
        return "UNKNOWN"
    return str(x)

def ffmpeg_exists():
    try:
        r = subprocess.run(["ffmpeg", "-version"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        return r.returncode == 0
    except Exception:
        return False

def ffprobe_exists():
    try:
        r = subprocess.run(["ffprobe", "-version"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        return r.returncode == 0
    except Exception:
        return False

if not ffmpeg_exists():
    raise RuntimeError("ffmpeg not found.")
if not ffprobe_exists():
    raise RuntimeError("ffprobe not found.")

def probe_video(video_path):
    cmd = [
        "ffprobe",
        "-v", "error",
        "-select_streams", "v:0",
        "-show_entries", "stream=width,height,avg_frame_rate,nb_frames",
        "-show_entries", "format=duration",
        "-of", "json",
        str(video_path)
    ]
    try:
        r = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        if r.returncode != 0:
            return None
        info = json.loads(r.stdout)
        if "streams" not in info or len(info["streams"]) == 0:
            return None
        s = info["streams"][0]

        def parse_rate(rate_str):
            if rate_str is None or rate_str == "0/0":
                return np.nan
            if "/" in rate_str:
                a, b = rate_str.split("/")
                a = float(a)
                b = float(b)
                return a / b if b != 0 else np.nan
            return float(rate_str)

        fps = parse_rate(s.get("avg_frame_rate", None))
        duration = info.get("format", {}).get("duration", None)
        duration = float(duration) if duration is not None else np.nan

        nb_frames = s.get("nb_frames", None)
        try:
            nb_frames = int(nb_frames) if nb_frames is not None else np.nan
        except:
            nb_frames = np.nan

        width = s.get("width", np.nan)
        height = s.get("height", np.nan)

        return {
            "fps": fps,
            "duration_sec": duration,
            "frame_count_reported": nb_frames,
            "width": width,
            "height": height,
        }
    except:
        return None

def make_dir(modality, distance, gesture, subject_id, pair_id):
    return FRAMES_ROOT / modality / safe_str(distance) / safe_str(gesture) / safe_str(subject_id) / safe_str(pair_id)

def count_frames(folder):
    if not folder.exists():
        return 0
    return len(list(folder.glob("frame_*.jpg")))

def trim_extract_ffmpeg(video_path, output_dir, trim_start=1.0, trim_end=1.0, jpeg_quality=95):
    """
    Trim start/end and extract all frames as jpg.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    meta = probe_video(video_path)
    if meta is None:
        return {"status": "probe_failed", "frames_extracted": 0, "meta": None}

    duration = meta["duration_sec"]
    if pd.isna(duration):
        return {"status": "duration_missing", "frames_extracted": 0, "meta": meta}

    trimmed_duration = duration - trim_start - trim_end
    if trimmed_duration <= 0:
        return {"status": "trim_too_aggressive", "frames_extracted": 0, "meta": meta}

    existing = count_frames(output_dir)
    if SKIP_IF_ALREADY_DONE and existing > 0 and not FORCE_REEXTRACT:
        return {"status": "skipped_existing", "frames_extracted": existing, "meta": meta}

    if FORCE_REEXTRACT and output_dir.exists():
        shutil.rmtree(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

    output_pattern = str(output_dir / "frame_%06d.jpg")

    # q:v for mjpeg image quality: smaller is higher quality
    cmd = [
        "ffmpeg",
        "-y",
        "-ss", str(trim_start),
        "-i", str(video_path),
        "-t", str(trimmed_duration),
        "-q:v", "2",
        output_pattern
    ]

    r = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if r.returncode != 0:
        return {"status": "ffmpeg_failed", "frames_extracted": count_frames(output_dir), "meta": meta}

    extracted = count_frames(output_dir)
    return {"status": "success", "frames_extracted": extracted, "meta": meta}

def remove_background_folder(input_dir, output_dir, session=None):
    """
    Remove RGB background from all extracted frames using rembg.
    """
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    frame_paths = sorted(input_dir.glob("frame_*.jpg"))
    if len(frame_paths) == 0:
        return {"status": "no_input_frames", "frames_written": 0}

    existing = count_frames(output_dir)
    if SKIP_IF_ALREADY_DONE and existing == len(frame_paths) and not FORCE_REEXTRACT:
        return {"status": "skipped_existing", "frames_written": existing}

    if FORCE_REEXTRACT and output_dir.exists():
        shutil.rmtree(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

    written = 0

    for fp in frame_paths:
        out_fp = output_dir / fp.name

        if SKIP_IF_ALREADY_DONE and out_fp.exists() and not FORCE_REEXTRACT:
            written += 1
            continue

        img_bgr = cv2.imread(str(fp), cv2.IMREAD_COLOR)
        if img_bgr is None:
            continue

        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        try:
            # remove returns RGBA bytes or ndarray depending on usage
            result = remove(img_rgb, session=session)
            if isinstance(result, bytes):
                nparr = np.frombuffer(result, np.uint8)
                rgba = cv2.imdecode(nparr, cv2.IMREAD_UNCHANGED)
            else:
                rgba = result

            if rgba is None:
                continue

            # If alpha exists, composite on black background
            if len(rgba.shape) == 3 and rgba.shape[2] == 4:
                rgb = rgba[:, :, :3].astype(np.float32)
                alpha = rgba[:, :, 3:4].astype(np.float32) / 255.0
                bg = np.zeros_like(rgb)
                comp = rgb * alpha + bg * (1 - alpha)
                comp = comp.astype(np.uint8)
            else:
                comp = rgba[:, :, :3] if len(rgba.shape) == 3 else rgba
                if comp.ndim == 2:
                    comp = np.stack([comp] * 3, axis=-1)

            comp_bgr = cv2.cvtColor(comp, cv2.COLOR_RGB2BGR)
            ok = cv2.imwrite(str(out_fp), comp_bgr, [int(cv2.IMWRITE_JPEG_QUALITY), JPEG_QUALITY])
            if ok:
                written += 1

        except Exception:
            continue

    return {"status": "success", "frames_written": written}

# ============================================================
# 5. BUILD TASKS
# ============================================================
tasks = []
for _, row in df.iterrows():
    pair_id = safe_str(row["pair_id"])
    subject_id = safe_str(row["subject_id"])
    gesture = safe_str(row["gesture"])
    distance = safe_str(row["distance"])

    rgb_dir = make_dir("RGB", distance, gesture, subject_id, pair_id)
    nir_dir = make_dir("NIR", distance, gesture, subject_id, pair_id)
    rgb_bg_dir = make_dir("RGB_BGREM", distance, gesture, subject_id, pair_id)

    tasks.append({
        "pair_id": pair_id,
        "subject_id": subject_id,
        "gesture": gesture,
        "distance": distance,
        "rgb_video_path": row["rgb_video_path"],
        "nir_video_path": row["nir_video_path"],
        "rgb_dir": rgb_dir,
        "nir_dir": nir_dir,
        "rgb_bg_dir": rgb_bg_dir,
    })

print("Total paired tasks:", len(tasks))
display(pd.DataFrame(tasks).head())

# ============================================================
# 6. PROCESS ONE PAIR
# ============================================================
def process_pair(task):
    result_rows = []

    pair_id = task["pair_id"]
    subject_id = task["subject_id"]
    gesture = task["gesture"]
    distance = task["distance"]

    # -------- RGB trimmed extraction --------
    rgb_info = trim_extract_ffmpeg(
        video_path=task["rgb_video_path"],
        output_dir=task["rgb_dir"],
        trim_start=TRIM_START_SEC,
        trim_end=TRIM_END_SEC,
        jpeg_quality=JPEG_QUALITY,
    )

    rgb_meta = rgb_info["meta"] if rgb_info["meta"] is not None else {}
    result_rows.append({
        "pair_id": pair_id,
        "subject_id": subject_id,
        "gesture": gesture,
        "distance": distance,
        "modality": "RGB",
        "video_path": task["rgb_video_path"],
        "output_dir": str(task["rgb_dir"]),
        "status": rgb_info["status"],
        "frames_extracted": rgb_info["frames_extracted"],
        "fps": rgb_meta.get("fps", np.nan),
        "duration_original_sec": rgb_meta.get("duration_sec", np.nan),
        "trim_start_sec": TRIM_START_SEC,
        "trim_end_sec": TRIM_END_SEC,
        "duration_after_trim_sec": (
            rgb_meta.get("duration_sec", np.nan) - TRIM_START_SEC - TRIM_END_SEC
            if rgb_info["meta"] is not None and pd.notna(rgb_meta.get("duration_sec", np.nan))
            else np.nan
        ),
        "width": rgb_meta.get("width", np.nan),
        "height": rgb_meta.get("height", np.nan),
    })

    # -------- NIR trimmed extraction --------
    nir_info = trim_extract_ffmpeg(
        video_path=task["nir_video_path"],
        output_dir=task["nir_dir"],
        trim_start=TRIM_START_SEC,
        trim_end=TRIM_END_SEC,
        jpeg_quality=JPEG_QUALITY,
    )

    nir_meta = nir_info["meta"] if nir_info["meta"] is not None else {}
    result_rows.append({
        "pair_id": pair_id,
        "subject_id": subject_id,
        "gesture": gesture,
        "distance": distance,
        "modality": "NIR",
        "video_path": task["nir_video_path"],
        "output_dir": str(task["nir_dir"]),
        "status": nir_info["status"],
        "frames_extracted": nir_info["frames_extracted"],
        "fps": nir_meta.get("fps", np.nan),
        "duration_original_sec": nir_meta.get("duration_sec", np.nan),
        "trim_start_sec": TRIM_START_SEC,
        "trim_end_sec": TRIM_END_SEC,
        "duration_after_trim_sec": (
            nir_meta.get("duration_sec", np.nan) - TRIM_START_SEC - TRIM_END_SEC
            if nir_info["meta"] is not None and pd.notna(nir_meta.get("duration_sec", np.nan))
            else np.nan
        ),
        "width": nir_meta.get("width", np.nan),
        "height": nir_meta.get("height", np.nan),
    })

    # -------- RGB background removal --------
    if USE_RGB_BACKGROUND_REMOVAL and REMBG_AVAILABLE:
        session = new_session("u2net")
        bg_info = remove_background_folder(
            input_dir=task["rgb_dir"],
            output_dir=task["rgb_bg_dir"],
            session=session
        )

        result_rows.append({
            "pair_id": pair_id,
            "subject_id": subject_id,
            "gesture": gesture,
            "distance": distance,
            "modality": "RGB_BGREM",
            "video_path": task["rgb_video_path"],
            "output_dir": str(task["rgb_bg_dir"]),
            "status": bg_info["status"],
            "frames_extracted": bg_info["frames_written"],
            "fps": rgb_meta.get("fps", np.nan),
            "duration_original_sec": rgb_meta.get("duration_sec", np.nan),
            "trim_start_sec": TRIM_START_SEC,
            "trim_end_sec": TRIM_END_SEC,
            "duration_after_trim_sec": (
                rgb_meta.get("duration_sec", np.nan) - TRIM_START_SEC - TRIM_END_SEC
                if rgb_info["meta"] is not None and pd.notna(rgb_meta.get("duration_sec", np.nan))
                else np.nan
            ),
            "width": rgb_meta.get("width", np.nan),
            "height": rgb_meta.get("height", np.nan),
        })

    return result_rows

# ============================================================
# 7. RUN ALL TASKS
# ============================================================
all_rows = []
start_time = time.time()

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(process_pair, task) for task in tasks]

    for fut in tqdm(as_completed(futures), total=len(futures), desc="Processing trimmed RGB + NIR pairs"):
        rows = fut.result()
        all_rows.extend(rows)

elapsed = time.time() - start_time

summary_df = pd.DataFrame(all_rows)
summary_df = summary_df.sort_values(["modality", "distance", "gesture", "subject_id", "pair_id"]).reset_index(drop=True)

# ============================================================
# 8. SAVE RESULTS
# ============================================================
summary_df.to_csv(SUMMARY_CSV, index=False)

summary_json = {
    "total_pairs_processed": int(len(tasks)),
    "total_summary_rows": int(len(summary_df)),
    "trim_start_sec": TRIM_START_SEC,
    "trim_end_sec": TRIM_END_SEC,
    "use_rgb_background_removal": bool(USE_RGB_BACKGROUND_REMOVAL),
    "status_counts": summary_df["status"].value_counts().to_dict() if len(summary_df) else {},
    "frames_by_modality": summary_df.groupby("modality")["frames_extracted"].sum().to_dict() if len(summary_df) else {},
    "avg_frames_per_video_by_modality": summary_df.groupby("modality")["frames_extracted"].mean().round(2).to_dict() if len(summary_df) else {},
    "elapsed_seconds": round(float(elapsed), 2),
    "elapsed_minutes": round(float(elapsed / 60.0), 2),
    "summary_csv": str(SUMMARY_CSV),
}

with open(SUMMARY_JSON, "w") as f:
    json.dump(summary_json, f, indent=4)

# ============================================================
# 9. REPORT
# ============================================================
print("\nSaved summary CSV to:", SUMMARY_CSV)
print("Saved summary JSON to:", SUMMARY_JSON)

print("\n========== FINAL SUMMARY ==========")
print(json.dumps(summary_json, indent=4))

print("\nStatus counts:")
display(summary_df["status"].value_counts())

print("\nFrames by modality:")
display(summary_df.groupby("modality")["frames_extracted"].sum())

print("\nAverage frames per video by modality:")
display(summary_df.groupby("modality")["frames_extracted"].mean().round(2))

print("\nSample rows:")
display(summary_df.head(12))

ROOT: /data/Sajjan_Singh/gesture_recognition
MANIFEST_PATH: /data/Sajjan_Singh/gesture_recognition/data/processed/manifests/paired_manifest_with_subjects.csv
FRAMES_ROOT: /data/Sajjan_Singh/gesture_recognition/data/processed/frames_trimmed
USE_RGB_BACKGROUND_REMOVAL: True
Rows to process: 749


,pair_id,pair_index_within_group,subject_id,gesture,distance,capture_id,rgb_video_path,nir_video_path,rgb_num_frames,nir_num_frames,rgb_fps,nir_fps,duration_rgb,duration_nir,fps_diff,frame_diff,duration_diff,is_valid_pair,rgb_width,rgb_height,nir_width,nir_height,rgb_filename,nir_filename,rgb_serial_no,nir_serial_no
0,PAIR_00001,1,4F_S001,doctor,4_feet,1,/data/Sajjan_Singh/gesture_recognition/data/raw/RGB/4_feet/doctor/video_20260303_154701.mp4,/data/Sajjan_Singh/gesture_recognition/data/raw/NIR/nir/4_feet/doctor/20260303154701830.mp4,184,135,29.696472,25.074294,6.196022,5.384,4.622178,49,0.812022,True,1080,1920,720,960,video_20260303_154701.mp4,20260303154701830.mp4,20260303154701,20260303154701830
1,PAIR_00002,2,4F_S002,doctor,4_feet,2,/data/Sajjan_Singh/gesture_recognition/data/raw/RGB/4_feet/doctor/video_20260303_155209.mp4,/data/Sajjan_Singh/gesture_recognition/data/raw/NIR/nir/4_feet/doctor/20260303155208958.mp4,173,137,30.019203,25.041126,5.762978,5.471,4.978077,36,0.291978,True,1080,1920,720,960,video_20260303_155209.mp4,20260303155208958.mp4,20260303155209,20260303155208958
2,PAIR_00003,3,4F_S003,doctor,4_feet,3,/data/Sajjan_Singh/gesture_recognition/data/raw/RGB/4_feet/doctor/video_20260303_160149.mp4,/data/Sajjan_Singh/gesture_recognition/data/raw/NIR/nir/4_feet/doctor/20260303160149432.mp4,180,137,30.019290,25.009127,5.996144,5.478,5.010163,43,0.518144,True,1080,1920,720,960,video_20260303_160149.mp4,20260303160149432.mp4,20260303160149,20260303160149432
3,PAIR_00004,4,4F_S004,doctor,4_feet,4,/data/Sajjan_Singh/gesture_recognition/data/raw/RGB/4_feet/doctor/video_20260303_161125.mp4,/data/Sajjan_Singh/gesture_recognition/data/raw/NIR/nir/4_feet/doctor/20260303161125449.mp4,175,142,30.019270,25.035261,5.829589,5.672,4.984009,33,0.157589,True,1080,1920,720,960,video_20260303_161125.mp4,20260303161125449.mp4,20260303161125,20260303161125449
4,PAIR_00005,5,4F_S005,doctor,4_feet,5,/data/Sajjan_Singh/gesture_recognition/data/raw/RGB/4_feet/doctor/video_20260303_161400.mp4,/data/Sajjan_Singh/gesture_recognition/data/raw/NIR/nir/4_feet/doctor/20260303161400573.mp4,167,135,30.019294,25.046382,5.563089,5.390,4.972912,32,0.173089,True,1080,1920,720,960,video_20260303_161400.mp4,20260303161400573.mp4,20260303161400,20260303161400573


Total paired tasks: 749


,pair_id,subject_id,gesture,distance,rgb_video_path,nir_video_path,rgb_dir,nir_dir,rgb_bg_dir
0,PAIR_00001,4F_S001,doctor,4_feet,/data/Sajjan_Singh/gesture_recognition/data/raw/RGB/4_feet/doctor/video_20260303_154701.mp4,/data/Sajjan_Singh/gesture_recognition/data/raw/NIR/nir/4_feet/doctor/20260303154701830.mp4,/data/Sajjan_Singh/gesture_recognition/data/processed/frames_trimmed/RGB/4_feet/doctor/4F_S001/PAIR_00001,/data/Sajjan_Singh/gesture_recognition/data/processed/frames_trimmed/NIR/4_feet/doctor/4F_S001/PAIR_00001,/data/Sajjan_Singh/gesture_recognition/data/processed/frames_trimmed/RGB_BGREM/4_feet/doctor/4F_S001/PAIR_00001
1,PAIR_00002,4F_S002,doctor,4_feet,/data/Sajjan_Singh/gesture_recognition/data/raw/RGB/4_feet/doctor/video_20260303_155209.mp4,/data/Sajjan_Singh/gesture_recognition/data/raw/NIR/nir/4_feet/doctor/20260303155208958.mp4,/data/Sajjan_Singh/gesture_recognition/data/processed/frames_trimmed/RGB/4_feet/doctor/4F_S002/PAIR_00002,/data/Sajjan_Singh/gesture_recognition/data/processed/frames_trimmed/NIR/4_feet/doctor/4F_S002/PAIR_00002,/data/Sajjan_Singh/gesture_recognition/data/processed/frames_trimmed/RGB_BGREM/4_feet/doctor/4F_S002/PAIR_00002
2,PAIR_00003,4F_S003,doctor,4_feet,/data/Sajjan_Singh/gesture_recognition/data/raw/RGB/4_feet/doctor/video_20260303_160149.mp4,/data/Sajjan_Singh/gesture_recognition/data/raw/NIR/nir/4_feet/doctor/20260303160149432.mp4,/data/Sajjan_Singh/gesture_recognition/data/processed/frames_trimmed/RGB/4_feet/doctor/4F_S003/PAIR_00003,/data/Sajjan_Singh/gesture_recognition/data/processed/frames_trimmed/NIR/4_feet/doctor/4F_S003/PAIR_00003,/data/Sajjan_Singh/gesture_recognition/data/processed/frames_trimmed/RGB_BGREM/4_feet/doctor/4F_S003/PAIR_00003
3,PAIR_00004,4F_S004,doctor,4_feet,/data/Sajjan_Singh/gesture_recognition/data/raw/RGB/4_feet/doctor/video_20260303_161125.mp4,/data/Sajjan_Singh/gesture_recognition/data/raw/NIR/nir/4_feet/doctor/20260303161125449.mp4,/data/Sajjan_Singh/gesture_recognition/data/processed/frames_trimmed/RGB/4_feet/doctor/4F_S004/PAIR_00004,/data/Sajjan_Singh/gesture_recognition/data/processed/frames_trimmed/NIR/4_feet/doctor/4F_S004/PAIR_00004,/data/Sajjan_Singh/gesture_recognition/data/processed/frames_trimmed/RGB_BGREM/4_feet/doctor/4F_S004/PAIR_00004
4,PAIR_00005,4F_S005,doctor,4_feet,/data/Sajjan_Singh/gesture_recognition/data/raw/RGB/4_feet/doctor/video_20260303_161400.mp4,/data/Sajjan_Singh/gesture_recognition/data/raw/NIR/nir/4_feet/doctor/20260303161400573.mp4,/data/Sajjan_Singh/gesture_recognition/data/processed/frames_trimmed/RGB/4_feet/doctor/4F_S005/PAIR_00005,/data/Sajjan_Singh/gesture_recognition/data/processed/frames_trimmed/NIR/4_feet/doctor/4F_S005/PAIR_00005,/data/Sajjan_Singh/gesture_recognition/data/processed/frames_trimmed/RGB_BGREM/4_feet/doctor/4F_S005/PAIR_00005


Processing trimmed RGB + NIR pairs:   0%|          | 0/749 [00:00<?, ?it/s]Downloading data from 'https://github.com/danielgatis/rembg/releases/download/v0.0.0/u2net.onnx' to file '/home/sajjan/.u2net/u2net.onnx'.




















































































































































































































































































































































































































































































































































































































































































































































































































...

modality
NIR          63449
RGB          82185
RGB_BGREM    82185
Name: frames_extracted, dtype: int64


Average frames per video by modality:


modality
NIR           84.71
RGB          109.73
RGB_BGREM    109.73
Name: frames_extracted, dtype: float64


Sample rows:


,pair_id,subject_id,gesture,distance,modality,video_path,output_dir,status,frames_extracted,fps,duration_original_sec,trim_start_sec,trim_end_sec,duration_after_trim_sec,width,height
0,PAIR_00001,4F_S001,doctor,4_feet,NIR,/data/Sajjan_Singh/gesture_recognition/data/raw/NIR/nir/4_feet/doctor/20260303154701830.mp4,/data/Sajjan_Singh/gesture_recognition/data/processed/frames_trimmed/NIR/4_feet/doctor/4F_S001/PAIR_00001,success,84,25.074294,5.384,1.0,1.0,3.384,720,960
1,PAIR_00002,4F_S002,doctor,4_feet,NIR,/data/Sajjan_Singh/gesture_recognition/data/raw/NIR/nir/4_feet/doctor/20260303155208958.mp4,/data/Sajjan_Singh/gesture_recognition/data/processed/frames_trimmed/NIR/4_feet/doctor/4F_S002/PAIR_00002,success,86,25.041126,5.471,1.0,1.0,3.471,720,960
2,PAIR_00003,4F_S003,doctor,4_feet,NIR,/data/Sajjan_Singh/gesture_recognition/data/raw/NIR/nir/4_feet/doctor/20260303160149432.mp4,/data/Sajjan_Singh/gesture_recognition/data/processed/frames_trimmed/NIR/4_feet/doctor/4F_S003/PAIR_00003,success,87,25.009127,5.478,1.0,1.0,3.478,720,960
3,PAIR_00004,4F_S004,doctor,4_feet,NIR,/data/Sajjan_Singh/gesture_recognition/data/raw/NIR/nir/4_feet/doctor/20260303161125449.mp4,/data/Sajjan_Singh/gesture_recognition/data/processed/frames_trimmed/NIR/4_feet/doctor/4F_S004/PAIR_00004,success,91,25.035261,5.672,1.0,1.0,3.672,720,960
4,PAIR_00005,4F_S005,doctor,4_feet,NIR,/data/Sajjan_Singh/gesture_recognition/data/raw/NIR/nir/4_feet/doctor/20260303161400573.mp4,/data/Sajjan_Singh/gesture_recognition/data/processed/frames_trimmed/NIR/4_feet/doctor/4F_S005/PAIR_00005,success,84,25.046382,5.390,1.0,1.0,3.390,720,960
5,PAIR_00006,4F_S006,doctor,4_feet,NIR,/data/Sajjan_Singh/gesture_recognition/data/raw/NIR/nir/4_feet/doctor/20260303161659874.mp4,/data/Sajjan_Singh/gesture_recognition/data/processed/frames_trimmed/NIR/4_feet/doctor/4F_S006/PAIR_00006,success,79,25.024248,5.155,1.0,1.0,3.155,720,960
6,PAIR_00007,4F_S007,doctor,4_feet,NIR,/data/Sajjan_Singh/gesture_recognition/data/raw/NIR/nir/4_feet/doctor/20260303162723511.mp4,/data/Sajjan_Singh/gesture_recognition/data/processed/frames_trimmed/NIR/4_feet/doctor/4F_S007/PAIR_00007,success,88,25.000000,5.520,1.0,1.0,3.520,720,960
7,PAIR_00008,4F_S008,doctor,4_feet,NIR,/data/Sajjan_Singh/gesture_recognition/data/raw/NIR/nir/4_feet/doctor/20260303163030280.mp4,/data/Sajjan_Singh/gesture_recognition/data/processed/frames_trimmed/NIR/4_feet/doctor/4F_S008/PAIR_00008,success,73,25.030525,4.914,1.0,1.0,2.914,720,960
8,PAIR_00009,4F_S009,doctor,4_feet,NIR,/data/Sajjan_Singh/gesture_recognition/data/raw/NIR/nir/4_feet/doctor/20260303163441824.mp4,/data/Sajjan_Singh/gesture_recognition/data/processed/frames_trimmed/NIR/4_feet/doctor/4F_S009/PAIR_00009,success,88,25.013594,5.517,1.0,1.0,3.517,720,960
9,PAIR_00010,4F_S010,doctor,4_feet,NIR,/data/Sajjan_Singh/gesture_recognition/data/raw/NIR/nir/4_feet/doctor/20260303163947452.mp4,/data/Sajjan_Singh/gesture_recognition/data/processed/frames_trimmed/NIR/4_feet/doctor/4F_S010/PAIR_00010,success,80,24.961300,5.168,1.0,1.0,3.168,720,960
